# Gap #1 for all six models: calibration and threshold policies (Colab)

Runs the Gap #1 analysis behind the submitted supplemental PDF's **Figure 1** on every finished frozen-protocol run, for all six models. No training, no GPU-heavy work.

**Method: unchanged from Figure 1.** The submitted PDF can't change, so neither can its method:
- One temperature per run and three threshold policies (0.5 default, Youden's J, F1-optimal) are fit on the **official validation set** (234 images, single centre crop) and applied **once** to the cached held-out test predictions (`<run>.csv`, written by `engine.py`'s test step).
- The functions are copied verbatim from `ArkPlus_Gaps_Colab.ipynb`. Section 5 re-derives Ark+'s published Figure 1 bars and **stops** if they don't match.
- These are the **validation-tuned** numbers. The `*_results.txt` files from `engine.py` report F1 / sensitivity / accuracy / MCC with thresholds fit on the *test* labels, which is what the submitted Table 3 shows. Keep the two kinds in separate, labelled columns, and never mix them.

**Before running**
- Run it from the **same Google account as the training notebooks**: the paths below are that account's My Drive.
- A run appears only once its tier has finished training *and* testing. Re-run this notebook as more models finish.
- Ark+'s **screening** tier is excluded. Its Drive checkpoint and predictions were replaced by an accidental resume on 2026-08-05; the clean original is on the Mac in `checkpoints/arkplus/`. Figure 1 uses only the headline tier.

**Output:** `chexpert_protocol_v1/gap1/gap1_all_models.csv` on Drive. Copy it into `results/gap1/` on the Mac, because `results/` is the source of truth for every number in the paper.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Colab's pre-installed stack (pandas, etc.) is compiled against numpy>=2's actual C ABI --
# this isn't just a pip-metadata preference, running them against numpy<2 crashes for real
# ("numpy.dtype size changed, may indicate binary incompatibility"). We previously tried
# downgrading numpy to dodge an *unrelated* numpy-2.0 incompatibility in imgaug (a transitive
# dep of albumentations) -- that broke pandas instead, and the downgrade didn't even take
# effect this-session anyway, since Colab pre-imports numpy before any of our cells run. So:
# make sure numpy is restored to >=2 here (undoing any earlier accidental downgrade left on
# disk), and fix the imgaug incompatibility directly instead (see section 2, patches 4 & 5).
import sys
!{sys.executable} -m pip install -q --break-system-packages "numpy>=2"
import numpy
print('numpy', numpy.__version__, '@', numpy.__file__)
print('kernel python:', sys.executable)

## 1. Dataset
Same Kaggle CheXpert-small download as every training notebook. Only the official validation images are read here; the held-out test predictions are already cached on Drive.

In [ ]:
import os, json, getpass, sys

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
del kaggle_key

!{sys.executable} -m pip install -q --break-system-packages kaggle
os.makedirs('/content/data', exist_ok=True)

# Using the kaggle package's own Python API instead of any CLI form -- it turns out this
# version of the package has no `__main__.py` (`-m kaggle` fails with "cannot be directly
# executed"), and the bare `kaggle` console script hits the same PATH-mismatch class of
# issue we saw with pip earlier. Calling the API in-process sidesteps all of that.
import kaggle
kaggle.api.authenticate()
# quiet=False -- dataset_download_files defaults to quiet=True, which is why this ran with
# zero output before. With it off, kaggle's own tqdm byte-progress bar (MB downloaded, speed,
# ETA) shows up instead of the download appearing to hang silently for several minutes.
kaggle.api.dataset_download_files('ashery/chexpert', path='/content/data', unzip=True, quiet=False)
print('Download complete.')

In [ ]:
# The train.csv/valid.csv paths are prefixed with 'CheXpert-v1.0-small/', but the
# Kaggle extract puts train/ and valid/ directly under /content/data. Symlink to bridge it.
import os
data_root = '/content/data'
link = os.path.join(data_root, 'CheXpert-v1.0-small')
if not os.path.islink(link) and not os.path.exists(link):
    os.symlink(data_root, link)

print(sorted(os.listdir(data_root)))
import pandas as pd
df = pd.read_csv(os.path.join(data_root, 'train.csv'))
print(df.shape, list(df.columns))

## 2. Training code and dependencies
Copied verbatim from the training notebooks: the same pinned packages and the same `Finetuning/` code from Drive, so models are rebuilt exactly as they were trained.

In [ ]:
import os, sys

# Install the same dependencies the training code needs (this doesn't change: copying our own
# code below doesn't install third-party packages).
!{sys.executable} -m pip install -q --break-system-packages timm==0.5.4 albumentations==0.4.3 opencv-python-headless scikit-image scikit-learn \
    simpleitk scipy pydicom yacs einops transformers pretrainedmodels imgaug pillow tqdm

# imgaug (a transitive dep of albumentations==0.4.3, old and unmaintained) has a np.sctypes
# reference (removed in numpy 2.0) in parameters.py, plus a dead `from scipy import misc` import
# (removed in modern scipy) in the same file. Both crash on *import*, so we can't `import imgaug`
# first to locate the file -- that import is exactly what crashes. Using importlib.util.find_spec
# instead, which locates the module without executing it.
import importlib.util, os as _os
_ia_dir = _os.path.dirname(importlib.util.find_spec('imgaug').origin)
_ia_params = _os.path.join(_ia_dir, 'parameters.py')
_s = open(_ia_params).read()
_before = _s
_s = _s.replace('np.sctypes["float"]', '[np.float16, np.float32, np.float64]')
_s = _s.replace('from scipy import misc\n', '')
if _s != _before:
    open(_ia_params, 'w').write(_s)
    print('imgaug patched:', _ia_params)
else:
    print('imgaug already patched (or patterns not found), skipping:', _ia_params)
import imgaug
print('imgaug import OK, version', imgaug.__version__)

# The actual CheXpert training code (dataloader/trainer/engine/models) is pulled straight from
# Drive instead of re-deriving ~15 sed/python patches against a live jlianglab/Ark clone every
# run. This copy already has everything: base compatibility fixes, all 6 models wired in, and
# the full frozen protocol (320px support for every architecture, patient-level held-out test
# split, stratified ablation subset, ECE metric, configurable AMP dtype, per-trial seeding) --
# kept in sync with the canonical copy at github.com/fei7fei/chexpert-baselines. Every notebook
# reads from this exact same folder, so the underlying code is guaranteed byte-identical across
# all 6 models -- required for the frozen-protocol comparison to be valid.
from google.colab import drive
drive.mount('/content/drive')

_DRIVE_CODE_SRC = '/content/drive/MyDrive/Project/models/Repos/ark_plus_repo/Ark_Plus/Finetuning'
if not os.path.exists('/content/chexpert_code'):
    import shutil
    shutil.copytree(_DRIVE_CODE_SRC, '/content/chexpert_code')
%cd /content/chexpert_code
print('Loaded pre-patched CheXpert training code (frozen protocol v1) from Drive.')

## 3. Settings, model registry, validation set

In [ ]:
%cd /content/chexpert_code
import sys, os
sys.path.insert(0, '.')

_old_argv = sys.argv
sys.argv = ['main_classification.py']
from main_classification import get_args_parser
args = get_args_parser()
sys.argv = _old_argv

# Same values as every training notebook's config cell (frozen protocol v1) -- only what inference needs.
INPUT_SIZE = 320
SEEDS_SCREENING = [13]
SEEDS_HEADLINE = [13, 42]
args.input_size = INPUT_SIZE
args.img_size = INPUT_SIZE
args.num_class = 14
args.data_set = "CheXpert"
args.data_dir = "/content/data"
args.uncertain_label = "Zeros"
args.device = "cuda"
args.batch_size = 64   # Figure 1's validation loader used Ark+'s args.batch_size (64)

diseases_14 = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
               'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
               'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
competition_5 = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
test_diseases_5 = [diseases_14.index(c) for c in competition_5]

from dataloader import CheXpert, build_transform_classification
official_val_csv = os.path.join(args.data_dir, "valid.csv")
dataset_val = CheXpert(images_path=args.data_dir, file_path=official_val_csv,
    augment=build_transform_classification(normalize=args.normalization, mode="valid", crop_size=args.input_size, resize=args.img_size),
    uncertain_label=args.uncertain_label, unknown_label=args.unknown_label)
assert len(dataset_val) == 234, f"official valid.csv should have 234 images, found {len(dataset_val)}"

# One entry per model: how to rebuild its architecture ("init" only builds an empty scaffold -- the
# fine-tuned weights are loaded from the checkpoint), the AMP dtype it was trained with, and where its
# runs live. Drive folders and experiment names match each training notebook and Results_Dashboard_Colab.ipynb.
_D = '/content/drive/MyDrive'
MODELS = [
    {"model": "Ark+",            "model_name": "swin_large",      "init": "random",       "amp_dtype": "fp16",
     "dir": f"{_D}/ark_plus_chexpert_checkpoints",    "prefix": "_protocol1_arkplus",     "tiers": ["headline"]},  # screening: see note above
    {"model": "SwinL (control)", "model_name": "swin_large",      "init": "random",       "amp_dtype": "fp16",
     "dir": f"{_D}/ark_plus_chexpert_checkpoints",    "prefix": "_protocol1_swinl",       "tiers": ["screening", "headline"]},
    {"model": "SwinB (V2)",      "model_name": "swinv2_base",     "init": "imagenet_21k", "amp_dtype": "bf16",
     "dir": f"{_D}/swinb_chexpert_checkpoints",       "prefix": "_protocol1_swinv2b",     "tiers": ["screening", "headline"]},
    {"model": "DenseNet-121",    "model_name": "densenet121",     "init": "random",       "amp_dtype": "fp16",
     "dir": f"{_D}/densenet121_chexpert_checkpoints", "prefix": "_protocol1_densenet121", "tiers": ["screening", "headline"]},
    {"model": "RadDino",         "model_name": "rad_dino",        "init": "rad_dino",     "amp_dtype": "fp16",
     "dir": f"{_D}/raddino_chexpert_checkpoints",     "prefix": "_protocol1_raddino",     "tiers": ["screening", "headline"]},
    {"model": "ConvNeXtV2",      "model_name": "convnextv2_base", "init": "imagenet_22k", "amp_dtype": "fp16",
     "dir": f"{_D}/convnextv2_chexpert_checkpoints",  "prefix": "_protocol1_convnextv2",  "tiers": ["screening", "headline"]},
]
SEEDS = {"screening": SEEDS_SCREENING, "headline": SEEDS_HEADLINE}
GAP1_OUT_DIR = f"{_D}/chexpert_protocol_v1/gap1"
os.makedirs(GAP1_OUT_DIR, exist_ok=True)
print(f"validation images: {len(dataset_val)} | models registered: {len(MODELS)}")


## 4. Gap #1 for every finished run
Fits temperature and the three threshold policies on validation only, then applies them once to the cached held-out test predictions. Runs that haven't finished are skipped.

In [ ]:
## Gap #1: calibration + per-label operating-point analysis (post-hoc, no training needed)
##
## engine.py's test-time meanF1/meanRecall/meanAccuracy (see the "start testing" block) pick
## each class's Youden's-J-optimal threshold FROM the held-out test set's own labels, then
## immediately score that same set with it -- an optimistic leak, the same category of bug as
## the checkpoint-selection leak fixed earlier this session (SESSION_NOTES.md bug #1), just for
## thresholds instead of epochs. AUC is unaffected (threshold-independent); F1/Recall/Accuracy
## as currently reported by the base pipeline are not, and shouldn't be treated as final numbers.
##
## This cell fits temperature (calibration) and three threshold policies on VALIDATION only, then
## applies them ONCE to the held-out test set -- no leak, by construction. Per PROTOCOL.md this
## is Youden's J; the earlier gap-analysis plan doc used per-class F1-optimal instead -- both are
## reported here alongside the naive 0.5 default, so the spread across policies is itself part of
## the result rather than a hidden methodology choice (matches Experiment Plan v3 SS5.2(b)).
##
## Reports each seed individually, not pooled -- with 2 headline seeds, a mean+/-range would
## misrepresent a 2-point range as an error bar (Experiment Plan v3 SS6.3).

import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_curve, f1_score, recall_score, accuracy_score
import torch
from torch.utils.data import DataLoader

from models import build_classification_model
from trainer import test_model
from utils import read_from_csv, meanECE


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def _sigmoid(x):
    return 1 / (1 + np.exp(-x))


def fit_temperature(y_val, p_val):
    # Single scalar T (one per model, not per-class) minimizing NLL on validation -- standard
    # temperature scaling (Guo et al., ICML 2017). p_val is post-sigmoid (test_model's output),
    # so logits are recovered via inverse-sigmoid rather than re-deriving raw logits from scratch.
    logits = _logit(p_val)

    def nll(T):
        T = max(T, 1e-3)
        p = np.clip(_sigmoid(logits / T), 1e-7, 1 - 1e-7)
        return -np.mean(y_val * np.log(p) + (1 - y_val) * np.log(1 - p))

    res = minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded")
    return res.x


def per_label_thresholds(y_val, p_val, mode):
    thresholds = np.full(y_val.shape[1], 0.5)
    for i in range(y_val.shape[1]):
        if not np.any(y_val[:, i]):
            continue
        fpr, tpr, thr = roc_curve(y_val[:, i], p_val[:, i])
        if mode == "youden":
            thresholds[i] = thr[np.argmax(tpr - fpr)]
        elif mode == "f1":
            f1s = [f1_score(y_val[:, i], (p_val[:, i] > t).astype(int)) for t in thr]
            thresholds[i] = thr[int(np.argmax(f1s))]
    return thresholds


def apply_thresholds(y, p, thresholds):
    sens, f1s, accs = [], [], []
    for i in range(y.shape[1]):
        if not np.any(y[:, i]):
            continue
        pred = (p[:, i] > thresholds[i]).astype(int)
        sens.append(recall_score(y[:, i], pred))
        f1s.append(f1_score(y[:, i], pred))
        accs.append(accuracy_score(y[:, i], pred))
    return float(np.mean(sens)), float(np.mean(f1s)), float(np.mean(accs))


class _ScaffoldArgs:
    pass


def load_trained_model(entry, exp_dir, run_name):
    # Same as ArkPlus_Gaps_Colab.ipynb's load_trained_model, except the architecture comes from the
    # registry instead of being hardcoded to swin_large.
    ckpt_path = os.path.join(exp_dir, run_name + ".pth.tar")
    if not os.path.exists(ckpt_path):
        return None
    _a = _ScaffoldArgs()
    _a.model_name = entry["model_name"]
    _a.init = entry["init"]     # scaffold only -- real (fine-tuned) weights loaded below
    _a.pretrained_weights = None
    _a.input_size = INPUT_SIZE
    _a.img_size = INPUT_SIZE
    _a.num_class = 14
    _model = build_classification_model(_a)
    _ckpt = torch.load(ckpt_path, map_location=args.device, weights_only=False)  # our own checkpoint
    _state_dict = _ckpt["state_dict"]
    _state_dict = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in _state_dict.items()}
    _model.load_state_dict(_state_dict, strict=True)
    _model.to(args.device)
    _model.eval()
    return _model


EXPECTED_TEST_ROWS = 33055  # held-out test images (Ark+'s gt.csv); any other count means a different test set

gap1_rows = []
gap1_raw = {}  # (model, run) -> arrays/values needed for the plotting cell below
for _entry in MODELS:
    args.amp_dtype = _entry["amp_dtype"]  # validation inference in the precision the model was trained in
    for _tier in _entry["tiers"]:
        _exp = f"{_entry['prefix']}_{_tier}"
        _exp_dir = os.path.join(_entry["dir"], "Models/Classification/CheXpert", _exp)
        for _i, _seed in enumerate(SEEDS[_tier]):
            _run = f"{_exp}_run_{_i}"
            _pred_csv = os.path.join(_exp_dir, _run + ".csv")
            _gt_csv = os.path.join(_exp_dir, "gt.csv")
            if not (os.path.exists(_pred_csv) and os.path.exists(_gt_csv)):
                print(f"Skipping {_entry['model']} / {_run} -- no cached test predictions yet (not finished training + testing).")
                continue

            _model = load_trained_model(_entry, _exp_dir, _run)
            if _model is None:
                print(f"Skipping {_entry['model']} / {_run} -- no checkpoint found at {_exp_dir}/{_run}.pth.tar")
                continue

            print(f"Evaluating {_entry['model']} / {_run} ...")
            _val_loader = DataLoader(dataset_val, batch_size=args.batch_size, shuffle=False, num_workers=2)
            y_val_full, p_val_full = test_model(_model, _val_loader, args)
            y_val = y_val_full.cpu().numpy()[:, test_diseases_5]
            p_val = p_val_full.cpu().numpy()[:, test_diseases_5]
            del _model
            torch.cuda.empty_cache()

            y_test = read_from_csv(_gt_csv)
            p_test = read_from_csv(_pred_csv)
            assert y_test.shape == (EXPECTED_TEST_ROWS, 5) and p_test.shape == y_test.shape, \
                f"{_run}: cached test arrays are {y_test.shape} / {p_test.shape}, expected ({EXPECTED_TEST_ROWS}, 5)"

            # ---- from here on, the same steps as ArkPlus_Gaps_Colab.ipynb cell 21 ----
            # Calibration: fit T on val, report ECE before/after on test
            T = fit_temperature(y_val, p_val)
            p_test_calibrated = _sigmoid(_logit(p_test) / T)
            ece_before, _ = meanECE(y_test, p_test)
            ece_after, _ = meanECE(y_test, p_test_calibrated)

            # Three threshold policies, all fit on VAL, applied once to TEST
            thr_050 = np.full(len(test_diseases_5), 0.5)
            thr_youden = per_label_thresholds(y_val, p_val, mode="youden")
            thr_f1 = per_label_thresholds(y_val, p_val, mode="f1")

            gap1_raw[(_entry["model"], _run)] = {
                "model": _entry["model"], "tier": _tier, "seed": _seed,
                "y_test": y_test, "p_test": p_test, "p_test_calibrated": p_test_calibrated,
                "T": T, "ece_before": float(ece_before), "ece_after": float(ece_after),
                "thresholds": {
                    "0.5 default": thr_050,
                    "Youden's J": thr_youden,
                    "F1-optimal": thr_f1,
                },
            }

            gap1_rows.append({
                "model": _entry["model"], "tier": _tier, "seed": _seed, "policy": "(calibration)",
                "sensitivity": None, "F1": None, "accuracy": None,
                "ECE_before": round(float(ece_before), 4), "ECE_after": round(float(ece_after), 4),
                "T": round(float(T), 3),
            })

            for _policy_name, _thr in [
                ("0.5 default", thr_050),
                ("Youden's J (val-tuned)", thr_youden),
                ("F1-optimal (val-tuned)", thr_f1),
            ]:
                sens, f1, acc = apply_thresholds(y_test, p_test, _thr)
                gap1_rows.append({
                    "model": _entry["model"], "tier": _tier, "seed": _seed, "policy": _policy_name,
                    "sensitivity": round(sens, 4), "F1": round(f1, 4), "accuracy": round(acc, 4),
                    "ECE_before": None, "ECE_after": None, "T": None,
                })

gap1_df = pd.DataFrame(gap1_rows)
print("\nPer-seed results (not pooled -- with 2 seeds, report the min-max range, never mean +/- std):")
display(gap1_df)


## 5. Check: does this reproduce the submitted Figure 1?
Must pass before any number from this notebook goes into the poster or manuscript.

In [ ]:
## Re-derive the submitted PDF's Figure 1 (Ark+ headline, accuracy and sensitivity per threshold policy,
## printed there to 2 decimals). If this notebook's method or inputs differed from what produced Figure 1,
## this cell fails -- and then none of the Gap #1 numbers above are comparable with the paper.
FIGURE1 = {  # (seed, policy) -> (accuracy, sensitivity), as printed in the submitted supplemental PDF, Figure 1
    (13, "0.5 default"): (0.86, 0.32), (13, "Youden's J (val-tuned)"): (0.66, 0.84), (13, "F1-optimal (val-tuned)"): (0.72, 0.73),
    (42, "0.5 default"): (0.86, 0.29), (42, "Youden's J (val-tuned)"): (0.67, 0.81), (42, "F1-optimal (val-tuned)"): (0.72, 0.73),
}
_TOL = 0.0051  # half a unit in the 2nd decimal, plus float slack
_mismatch = []
for (_seed, _policy), (_acc, _sens) in FIGURE1.items():
    _row = gap1_df[(gap1_df["model"] == "Ark+") & (gap1_df["tier"] == "headline")
                   & (gap1_df["seed"] == _seed) & (gap1_df["policy"] == _policy)] if len(gap1_df) else gap1_df
    if len(_row) != 1:
        _mismatch.append(f"seed {_seed} / {_policy}: no Ark+ headline row (is its run on Drive?)")
        continue
    _a, _s = float(_row["accuracy"].iloc[0]), float(_row["sensitivity"].iloc[0])
    _ok = abs(_a - _acc) <= _TOL and abs(_s - _sens) <= _TOL
    print(f"seed {_seed:>2}  {_policy:<24} accuracy {_a:.4f} (Fig. 1: {_acc:.2f})   sensitivity {_s:.4f} (Fig. 1: {_sens:.2f})   {'OK' if _ok else 'MISMATCH'}")
    if not _ok:
        _mismatch.append(f"seed {_seed} / {_policy}: got accuracy {_a:.4f}, sensitivity {_s:.4f}")
assert not _mismatch, "Figure 1 NOT reproduced -- do not use these Gap #1 numbers:\n" + "\n".join(_mismatch)
print("\nFigure 1 reproduced: same method and inputs as the submitted PDF.")


## 6. Save

In [ ]:
_out = os.path.join(GAP1_OUT_DIR, "gap1_all_models.csv")
gap1_df.to_csv(_out, index=False)
print(f"Saved {_out} ({len(gap1_df)} rows, {gap1_df['model'].nunique() if len(gap1_df) else 0} models).")
print("Copy it into results/gap1/ on the Mac -- results/ is the source of truth for every number in the paper.")


## 7. Charts

In [ ]:
## Figure 1-style charts for every model (headline tier), plus reliability diagrams.
## Same colours and layout as ArkPlus_Gaps_Colab.ipynb cell 22; one row per model, one column per seed.
import matplotlib.pyplot as plt
import numpy as np

_C_ACC = "#0DA0BE"
_C_SENS = "#8D55AA"
_C_MUTED = "#5C6773"

_headline = {k: v for k, v in gap1_raw.items() if v["tier"] == "headline"}
_models = [m["model"] for m in MODELS if any(v["model"] == m["model"] for v in _headline.values())]
if not _headline:
    print("No headline runs with data yet -- nothing to plot.")
else:
    _ncol = len(SEEDS_HEADLINE)
    figA, axesA = plt.subplots(len(_models), _ncol, figsize=(5.5 * _ncol, 3.8 * len(_models)), squeeze=False)
    figB, axesB = plt.subplots(len(_models), _ncol, figsize=(5.5 * _ncol, 4.4 * len(_models)), squeeze=False)
    for _r, _m in enumerate(_models):
        for _c, _seed in enumerate(SEEDS_HEADLINE):
            ax2, ax = axesA[_r, _c], axesB[_r, _c]
            _d = next((v for v in _headline.values() if v["model"] == _m and v["seed"] == _seed), None)
            if _d is None:
                ax2.axis("off"); ax.axis("off")
                continue

            # --- accuracy vs sensitivity, by threshold policy (the submitted Figure 1's chart) ---
            _policies = list(_d["thresholds"].keys())
            _sens_vals, _acc_vals = [], []
            for _pname, _thr in _d["thresholds"].items():
                _s, _, _a = apply_thresholds(_d["y_test"], _d["p_test"], _thr)
                _sens_vals.append(_s); _acc_vals.append(_a)
            _x = np.arange(len(_policies))
            _w = 0.35
            _b1 = ax2.bar(_x - _w / 2, _acc_vals, _w, label="Accuracy", color=_C_ACC)
            _b2 = ax2.bar(_x + _w / 2, _sens_vals, _w, label="Sensitivity", color=_C_SENS)
            for _bars in (_b1, _b2):
                ax2.bar_label(_bars, fmt="%.2f", fontsize=8, padding=2)
            ax2.set_xticks(_x); ax2.set_xticklabels(_policies, fontsize=8)
            ax2.set_ylim(0, 1.05)
            ax2.set_ylabel("Score")
            ax2.set_title(f"{_m} headline, seed {_seed}", fontsize=10)
            ax2.legend(fontsize=8)
            ax2.spines[["top", "right"]].set_visible(False)

            # --- reliability diagram (pooled across the 5 competition classes) ---
            _bins = np.linspace(0, 1, 11)
            for _label, _p, _color, _style in [
                ("before calibration", _d["p_test"], _C_MUTED, "--"),
                (f"after (T={_d['T']:.2f})", _d["p_test_calibrated"], _C_SENS, "-"),
            ]:
                _y_flat, _p_flat = _d["y_test"].ravel(), _p.ravel()
                _bin_ids = np.clip(np.digitize(_p_flat, _bins) - 1, 0, 9)
                _xs, _ys = [], []
                for _b in range(10):
                    _mask = _bin_ids == _b
                    if _mask.sum() == 0:
                        continue
                    _xs.append(_p_flat[_mask].mean())
                    _ys.append(_y_flat[_mask].mean())
                ax.plot(_xs, _ys, _style, color=_color, linewidth=2, marker="o", markersize=4, label=_label)
            ax.plot([0, 1], [0, 1], ":", color=_C_MUTED, linewidth=1, alpha=0.6, label="perfect calibration")
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)
            ax.set_xlabel("Mean predicted probability")
            ax.set_ylabel("Empirical positive rate")
            ax.set_title(f"{_m} headline, seed {_seed}\nECE {_d['ece_before']:.4f} -> {_d['ece_after']:.4f}", fontsize=10)
            ax.legend(fontsize=8, loc="upper left")
            ax.spines[["top", "right"]].set_visible(False)
    figA.suptitle("Accuracy vs sensitivity by threshold policy (fit on validation, applied once to held-out test)")
    figA.tight_layout()
    figB.suptitle("Reliability before / after temperature scaling (held-out test)")
    figB.tight_layout()
    plt.show()
